# Notebook 04 — LLM coding of OE responses

**Project:** Study 2 — Chatbot Neuroticism (DGK Lab)  
**Pipeline stage:** 4 of 8 (NLP Colab notebooks) + 1 integration Rmd  
**Runtime:** ~45–90 minutes (sequential across 3 LLMs)  
**API costs:** ~$8–12 total across all 3 providers

## Purpose

Apply the locked rubric (`rubric_v2.json`) to every non-blank OE response in the analytic sample, using three LLMs from three different families:

| LLM | Family | Role |
|---|---|---|
| Claude Sonnet 4.6   | Anthropic | Mid-tier Anthropic coder; pairs with Opus's role as rubric author |
| GPT-5.4-mini        | OpenAI    | Mid-tier OpenAI coder; cost-efficient on constrained classification |
| Gemini 3 Flash      | Google    | Mid-tier Google coder; fast, low-cost |

Each (response × LLM) pair produces a structured set of codes — one value per *category* (binary categories return a single code_id; multilabel categories return a list of code_ids). These are then expanded to per-theme binary indicators in long form.

## Why three LLMs

Cross-family agreement is the reliability metric for the manuscript. Three Anthropic models would share systematic biases. Three families gives us defensible κ values that survive review. The rubric was authored by Opus 4.7 (see Notebook 03) — that asymmetry is disclosed in the manuscript. We use **mid-tier** models for coding (rather than top-tier) because the task is constrained classification against a locked rubric with anchors: top-tier capability is not the bottleneck, and mid-tier saves ~$20 without compromising the cross-family argument.

## Caching

Every (response_hash × rubric_hash × llm × oe_field) combination is cached to disk. If the notebook crashes halfway, restarting picks up where it left off without re-burning tokens. Editing the rubric and re-running invalidates only the affected cache entries.

## Inputs

- `data/derived/rubric/rubric_v2.json` (from Notebook 03, optionally edited by you and Alice)
- `data/derived/open_ends_long.parquet` (from Notebook 01)

## Outputs

- `data/derived/coding/oe_coded_long.parquet` — one row per (sona_id × oe_field × code_id × llm)
- `data/derived/coding/oe_coded_per_llm/{llm_id}.parquet` — per-LLM raw category outputs
- `data/derived/coding/_cache/` — content-hashed call cache (kept across runs)
- `data/derived/coding/coding_manifest.json` — costs, tokens, what was coded

## API keys required

Set these as Colab Secrets (left sidebar > key icon) using these exact names:

- `ANTHROPIC_API_KEY`
- `CharacterAI_OpenAI`
- `CharacterAI_Gemini` (Google AI Studio key — make at https://aistudio.google.com/apikey)


## 1. Setup imports

In [ ]:
import sys, os, json, time, hashlib, random
from pathlib import Path
from datetime import datetime
from typing import List, Dict, Optional, Any, Tuple

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 60)
pd.set_option('display.max_colwidth', 200)

print(f'Python: {sys.version.split()[0]}')
print(f'pandas: {pd.__version__}')
print(f'Run timestamp: {datetime.now().isoformat(timespec="seconds")}')

In [ ]:
# Auto-install LLM SDKs
import subprocess

def ensure(pkg_import_name: str, pip_name: Optional[str] = None) -> None:
    try:
        __import__(pkg_import_name)
    except ImportError:
        target = pip_name or pkg_import_name
        print(f'Installing {target}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', target])

ensure('anthropic', 'anthropic>=0.40.0')
ensure('openai', 'openai>=1.50.0')
ensure('google.genai', 'google-genai>=0.3.0')
ensure('tqdm', 'tqdm')

import anthropic
import openai
from google import genai as google_genai
from tqdm.auto import tqdm

print(f'anthropic: {anthropic.__version__}')
print(f'openai:    {openai.__version__}')
print('google-genai installed.')

## 2. Project paths

In [ ]:
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    PROJECT_ROOT = Path('/content/drive/MyDrive/DGK Lab/Reasoning Lab/'
                        'DGK Lab - AI Training Data, CharacterAI, Tort Liability/'
                        'Analysis/Study 2')
else:
    PROJECT_ROOT = Path.cwd().parent if (Path.cwd().name == 'NLP') else Path.cwd()

DATA_DERIVED = PROJECT_ROOT / 'data' / 'derived'
RUBRIC_DIR   = DATA_DERIVED / 'rubric'
CODING_DIR   = DATA_DERIVED / 'coding'
PER_LLM_DIR  = CODING_DIR / 'oe_coded_per_llm'
CACHE_DIR    = CODING_DIR / '_cache'
OUTPUTS_NLP  = PROJECT_ROOT / 'outputs' / 'NLP'

for d in [CODING_DIR, PER_LLM_DIR, CACHE_DIR, OUTPUTS_NLP]:
    d.mkdir(parents=True, exist_ok=True)

PATH_OES = DATA_DERIVED / 'open_ends_long.parquet'
if not PATH_OES.exists():
    raise FileNotFoundError(f'Missing input from Notebook 01: {PATH_OES}')

print(f'Project root: {PROJECT_ROOT}')
print(f'Coding dir:   {CODING_DIR}')


## 3. Configuration

The model strings below are May-2026-current. If a model is deprecated by the time you run this, replace the string in the `LLMS` dict.

In [ ]:
# Which rubric version to use. Default to v2 (post-review). Change to 'v1' to run against original.
RUBRIC_VERSION = 'v2'

# LLM configurations: three mid-tier models from three families, all current-gen as of May 2026.
# Rationale: constrained classification against a locked rubric with anchors does not require
# top-tier capability. Cross-family agreement is the reliability argument, not within-family
# size. Mid-tier saves ~$20 over top-tier without compromising methodological standing.
LLMS = {
    'sonnet-4-6': {
        'family': 'anthropic',
        'model':  'claude-sonnet-4-6',
        'price_in_per_m':  3.00,
        'price_out_per_m': 15.00,
        'max_tokens_out':  3000,
    },
    'gpt-5-4-mini': {
        'family': 'openai',
        'model':  'gpt-5.4-mini',
        'price_in_per_m':  0.25,
        'price_out_per_m': 2.00,
        'max_tokens_out':  3000,
    },
    'gemini-3-flash': {
        'family': 'google',
        'model':  'gemini-3-flash-preview',   # Preview suffix is required as of May 2026
        'price_in_per_m':  0.50,              # Preview pricing: $0.50/M input
        'price_out_per_m': 3.00,              # Preview pricing: $3.00/M output (includes thinking tokens)
        'max_tokens_out':  3000,
    },
}

# Filter: only code substantive responses (>= 3 words)
MIN_WORDS_FOR_CODING = 3

# Pilot size: how many responses per (LLM x question) for the GO gate
PILOT_N_PER_QUESTION = 3

# Cost ceiling (USD) — total across all 3 LLMs
COST_CEILING_USD = 20.00

# Retries
MAX_RETRIES = 3
RETRY_BASE_DELAY = 2.0

# Reproducibility
RANDOM_SEED = 42

# Save partial progress every N successful calls
PARTIAL_SAVE_EVERY = 25

print(f'Rubric version: {RUBRIC_VERSION}')
print(f'LLMs: {list(LLMS.keys())}')
print(f'Cost ceiling (total): ${COST_CEILING_USD}')


## 4. API keys

In [ ]:
def get_secret(name: str) -> Optional[str]:
    if IN_COLAB:
        try:
            from google.colab import userdata
            return userdata.get(name)
        except Exception as e:
            print(f'[WARN] Could not read Colab Secret {name}: {e}')
    return os.environ.get(name)

ANTHROPIC_API_KEY = get_secret('ANTHROPIC_API_KEY')
OPENAI_API_KEY    = get_secret('CharacterAI_OpenAI')
GOOGLE_API_KEY    = get_secret('CharacterAI_Gemini')

for name, key in [('ANTHROPIC_API_KEY',    ANTHROPIC_API_KEY),
                  ('CharacterAI_OpenAI',   OPENAI_API_KEY),
                  ('CharacterAI_Gemini',   GOOGLE_API_KEY)]:
    if key is None:
        print(f'[ERROR] {name} not found.')
    else:
        print(f'[OK] {name} loaded (length={len(key)})')

assert ANTHROPIC_API_KEY and OPENAI_API_KEY and GOOGLE_API_KEY, 'All three API keys required'

anthropic_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
openai_client    = openai.OpenAI(api_key=OPENAI_API_KEY)
google_client    = google_genai.Client(api_key=GOOGLE_API_KEY)

print('Clients initialized.')


## 5. Load and validate rubric

In [ ]:
rubric_path = RUBRIC_DIR / f'rubric_{RUBRIC_VERSION}.json'
if not rubric_path.exists():
    raise FileNotFoundError(f'Rubric not found: {rubric_path}. Run Notebook 03 first, '
                             f'or change RUBRIC_VERSION.')

with open(rubric_path) as f:
    rubric = json.load(f)

# Hash the rubric content (sorted JSON) so cache invalidates if the rubric changes
rubric_text = json.dumps(rubric, sort_keys=True, ensure_ascii=False)
rubric_hash = hashlib.sha256(rubric_text.encode('utf-8')).hexdigest()[:12]

print(f'Rubric loaded: {rubric_path.name}')
print(f'Rubric hash:   {rubric_hash}')
print(f'Created:       {rubric.get("created_at", "unknown")}')
print(f'Questions:     {list(rubric["questions"].keys())}')

# Validate structure
issues = []
for q, qdata in rubric['questions'].items():
    if not qdata.get('themes'):
        issues.append(f'{q}: no themes')
    if not qdata.get('categories'):
        issues.append(f'{q}: no categories')
    cat_ids = set(qdata.get('categories', {}).keys())
    for t in qdata.get('themes', []):
        if not t.get('code_id'):
            issues.append(f'{q}: theme missing code_id')
        if not t.get('definition'):
            issues.append(f'{q}/{t.get("code_id", "?")}: missing definition')
        if t.get('category') not in cat_ids:
            issues.append(f'{q}/{t.get("code_id", "?")}: category {t.get("category")} not in defined categories')

if issues:
    print('\n[VALIDATION ISSUES]')
    for x in issues:
        print(f'  - {x}')
    raise RuntimeError(f'Rubric has {len(issues)} structural issues. Fix before coding.')
else:
    print('\n[OK] Rubric structure validated.')

## 6. Load OE responses and build coding queue

In [ ]:
oes = pd.read_parquet(PATH_OES)
oes['n_words'] = oes['response'].fillna('').astype(str).str.split().str.len()

# Coding queue: only primary OEs in the rubric, only substantive responses
TARGET_QUESTIONS = list(rubric['questions'].keys())
queue = oes[
    (oes['oe_field'].isin(TARGET_QUESTIONS)) &
    (~oes['is_blank']) &
    (oes['n_words'] >= MIN_WORDS_FOR_CODING)
].copy().reset_index(drop=True)

# Compute response hash for caching
queue['response_hash'] = queue['response'].astype(str).map(
    lambda s: hashlib.sha256(s.encode('utf-8')).hexdigest()[:12]
)

print(f'Coding queue: {len(queue):,} responses')
print(f'  by question:')
for q, n in queue['oe_field'].value_counts().sort_index().items():
    n_low = ((queue['oe_field'] == q) & (queue['condition_label'] == 'LowN')).sum()
    n_high = ((queue['oe_field'] == q) & (queue['condition_label'] == 'HighN')).sum()
    print(f'    {q}: {n} (LowN={n_low}, HighN={n_high})')
print(f'\nTotal API calls planned: {len(queue) * len(LLMS):,} '
      f'({len(queue)} responses × {len(LLMS)} LLMs)')

## 7. Caching helpers

In [ ]:
def cache_key(llm_id: str, response_hash: str, oe_field: str) -> str:
    """Cache key includes rubric_hash so cache invalidates if rubric changes."""
    return f'{llm_id}__{rubric_hash}__{oe_field}__{response_hash}'

def cache_path(key: str) -> Path:
    return CACHE_DIR / f'{key}.json'

def cache_load(key: str) -> Optional[Dict[str, Any]]:
    p = cache_path(key)
    if not p.exists():
        return None
    try:
        return json.loads(p.read_text())
    except Exception as e:
        print(f'  [WARN] cache load failed {p.name}: {e}')
        return None

def cache_save(key: str, payload: Dict[str, Any]) -> None:
    cache_path(key).write_text(json.dumps(payload, ensure_ascii=False))

# Count existing cache hits
existing_keys = {p.stem for p in CACHE_DIR.glob('*.json')}
print(f'Cache: {len(existing_keys):,} existing entries')

## 8. Build the coding prompt

The prompt is constructed once per (response × question × llm). Same wording across all three LLMs so any cross-LLM disagreement reflects model differences, not prompt differences.

In [ ]:
CODING_PROMPT_TEMPLATE = '''You are coding a participant's response according to a pre-specified rubric. Apply the rubric strictly. Do not add codes that are not in the rubric.

QUESTION ASKED TO PARTICIPANT
"""{question_text}"""

PARTICIPANT'S RESPONSE
"""{response}"""

RUBRIC

Categories:
{categories_block}

Themes:
{themes_block}

CODING INSTRUCTIONS

1. For each CATEGORY, decide which themes apply to the response.
2. For BINARY categories (only one theme can apply): output the chosen theme's code_id, or null if none of the themes in that category apply.
3. For MULTILABEL categories (any number can apply): output a list of code_ids that apply, or [] if none apply.
4. Code based on what is explicitly or strongly implied by the response. Do not infer themes the participant did not actually invoke.
5. If the response is too vague or brief to support a category's coding, output null (binary) or [] (multilabel) for that category.
6. Use only the code_ids listed in the rubric above. Do not invent new codes.

OUTPUT FORMAT

Return a single JSON object with one key per category, plus two metadata fields:

{{
{category_keys_template}
  "_confidence": "high" | "medium" | "low",
  "_ambiguous_categories": ["list of category_ids that were genuinely hard to code, can be empty"]
}}

Output ONLY the JSON object. No preamble, no explanation, no markdown fences.'''

def build_prompt(question_id: str, response: str) -> str:
    qdata = rubric['questions'][question_id]

    # Categories block
    cat_lines = []
    for cat_id, cat_info in qdata['categories'].items():
        cat_lines.append(f'  - `{cat_id}` ({cat_info["coding_type"]}): '
                         f'{cat_info.get("label", "")}. {cat_info.get("description", "")}')
    categories_block = '\n'.join(cat_lines)

    # Themes block, grouped by category
    theme_lines = []
    for cat_id, cat_info in qdata['categories'].items():
        theme_lines.append(f'\n[Category: {cat_id}]')
        for t in qdata['themes']:
            if t.get('category') != cat_id:
                continue
            theme_lines.append(f'  Code: `{t["code_id"]}` — {t.get("label", "")}')
            theme_lines.append(f'    Definition: {t.get("definition", "")}')
            if t.get('positive_anchors'):
                theme_lines.append(f'    Examples that COUNT:')
                for a in t['positive_anchors'][:3]:
                    theme_lines.append(f'      • {a}')
            if t.get('negative_anchors'):
                theme_lines.append(f'    Examples that DO NOT count:')
                for a in t['negative_anchors'][:2]:
                    theme_lines.append(f'      • {a}')
    themes_block = '\n'.join(theme_lines)

    # Category keys template (shows the LLM what JSON shape to produce)
    cat_key_lines = []
    for cat_id, cat_info in qdata['categories'].items():
        if cat_info['coding_type'] == 'binary':
            cat_key_lines.append(f'  "{cat_id}": "<code_id>" or null,')
        else:
            cat_key_lines.append(f'  "{cat_id}": ["<code_id>", ...] or [],')
    category_keys_template = '\n'.join(cat_key_lines)

    return CODING_PROMPT_TEMPLATE.format(
        question_text=qdata['question_text'],
        response=response.replace('"""', '""'),
        categories_block=categories_block,
        themes_block=themes_block,
        category_keys_template=category_keys_template,
    )

# Spot-check
example_q = TARGET_QUESTIONS[0]
example_resp = queue[queue['oe_field'] == example_q]['response'].iloc[0]
example_prompt = build_prompt(example_q, example_resp)
print(f'Prompt for {example_q} on first response (first 1500 chars):')
print('-' * 80)
print(example_prompt[:1500])
print('...' if len(example_prompt) > 1500 else '')

## 9. Unified LLM caller

One function per family that takes a prompt and returns a normalized result dict (parsed JSON, token usage, cost). Each handles its family's quirks (system messages, JSON mode, response shape).

In [ ]:
def parse_json_response(text: str) -> Optional[Dict[str, Any]]:
    """Tolerant JSON parser — handles markdown fences and stray whitespace."""
    if not text:
        return None
    cleaned = text.strip()
    if cleaned.startswith('```'):
        lines = cleaned.split('\n')
        if lines[0].startswith('```'):
            lines = lines[1:]
        if lines and lines[-1].strip() == '```':
            lines = lines[:-1]
        cleaned = '\n'.join(lines)
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        start = cleaned.find('{')
        end = cleaned.rfind('}')
        if start >= 0 and end > start:
            try:
                return json.loads(cleaned[start:end+1])
            except json.JSONDecodeError:
                return None
        return None

def call_anthropic(prompt: str, model: str, max_tokens: int) -> Dict[str, Any]:
    # Note: Opus 4.7 locks temperature/top_p/top_k. Don't pass them.
    resp = anthropic_client.messages.create(
        model=model,
        max_tokens=max_tokens,
        messages=[{'role': 'user', 'content': prompt}],
    )
    text = resp.content[0].text if resp.content else ''
    return {
        'text': text,
        'parsed_json': parse_json_response(text),
        'input_tokens': resp.usage.input_tokens,
        'output_tokens': resp.usage.output_tokens,
    }

def call_openai(prompt: str, model: str, max_tokens: int) -> Dict[str, Any]:
    # GPT-5.x reasoning family typically only accepts default temperature.
    # Don't pass temperature; rely on response_format for structured output.
    resp = openai_client.chat.completions.create(
        model=model,
        messages=[{'role': 'user', 'content': prompt}],
        max_completion_tokens=max_tokens,
        response_format={'type': 'json_object'},
    )
    text = resp.choices[0].message.content or ''
    return {
        'text': text,
        'parsed_json': parse_json_response(text),
        'input_tokens': resp.usage.prompt_tokens,
        'output_tokens': resp.usage.completion_tokens,
    }

def call_google(prompt: str, model: str, max_tokens: int) -> Dict[str, Any]:
    resp = google_client.models.generate_content(
        model=model,
        contents=prompt,
        config={
            'temperature': 0.1,
            'max_output_tokens': max_tokens,
            'response_mime_type': 'application/json',
        },
    )
    text = resp.text or ''
    in_tok = getattr(resp.usage_metadata, 'prompt_token_count', None) or 0
    out_tok = getattr(resp.usage_metadata, 'candidates_token_count', None) or 0
    return {
        'text': text,
        'parsed_json': parse_json_response(text),
        'input_tokens': in_tok,
        'output_tokens': out_tok,
    }

CALLERS = {
    'anthropic': call_anthropic,
    'openai':    call_openai,
    'google':    call_google,
}

def call_llm(llm_id: str, prompt: str) -> Dict[str, Any]:
    """Unified caller with retries."""
    cfg = LLMS[llm_id]
    caller = CALLERS[cfg['family']]
    last_err = None
    for attempt in range(MAX_RETRIES):
        try:
            r = caller(prompt, cfg['model'], cfg['max_tokens_out'])
            in_cost  = r['input_tokens']  / 1e6 * cfg['price_in_per_m']
            out_cost = r['output_tokens'] / 1e6 * cfg['price_out_per_m']
            r['cost'] = in_cost + out_cost
            r['llm_id'] = llm_id
            return r
        except Exception as e:
            last_err = e
            wait = RETRY_BASE_DELAY ** (attempt + 1)
            if attempt < MAX_RETRIES - 1:
                time.sleep(wait)
    raise RuntimeError(f'{llm_id} failed after {MAX_RETRIES} retries: {last_err}')

print('LLM callers ready.')

## 10. Pilot pass

Code 3 responses per OE question per LLM (= 36 calls total) before committing to the full run. Verifies that all three families return parseable JSON in the expected shape, and gives you a chance to spot-check before spending real money.

In [ ]:
rng = np.random.RandomState(RANDOM_SEED)
pilot_jobs = []
for q in TARGET_QUESTIONS:
    sub = queue[queue['oe_field'] == q]
    n = min(PILOT_N_PER_QUESTION, len(sub))
    picks = sub.sample(n=n, random_state=rng)
    for _, row in picks.iterrows():
        pilot_jobs.append(row.to_dict())

print(f'Pilot jobs: {len(pilot_jobs)} responses × {len(LLMS)} LLMs = '
      f'{len(pilot_jobs) * len(LLMS)} calls')

pilot_results = []
total_pilot_cost = 0.0

for job in tqdm(pilot_jobs, desc='Pilot'):
    prompt = build_prompt(job['oe_field'], job['response'])
    for llm_id in LLMS:
        try:
            r = call_llm(llm_id, prompt)
            pilot_results.append({
                'sona_id': job['sona_id'],
                'oe_field': job['oe_field'],
                'condition_label': job['condition_label'],
                'llm_id': llm_id,
                'parsed': r['parsed_json'],
                'parse_ok': r['parsed_json'] is not None,
                'in_tok': r['input_tokens'],
                'out_tok': r['output_tokens'],
                'cost': r['cost'],
                'error': None,
            })
            total_pilot_cost += r['cost']
        except Exception as e:
            pilot_results.append({
                'sona_id': job['sona_id'],
                'oe_field': job['oe_field'],
                'condition_label': job['condition_label'],
                'llm_id': llm_id,
                'parsed': None,
                'parse_ok': False,
                'in_tok': 0,
                'out_tok': 0,
                'cost': 0.0,
                'error': str(e),
            })

pilot_df = pd.DataFrame(pilot_results)
print(f'\nPilot complete. Cost: ${total_pilot_cost:.4f}')
print(f'\nParse success rate by LLM:')
print(pilot_df.groupby('llm_id')['parse_ok'].agg(['sum', 'count', 'mean']).round(3))

# Surface any error messages from failed calls so we can diagnose
errors = pilot_df[pilot_df['error'].notna() & (pilot_df['error'] != '')]
if len(errors) > 0:
    print(f'\n[DIAGNOSTIC] {len(errors)} pilot calls hit errors. Sample messages:')
    for llm_id, grp in errors.groupby('llm_id'):
        print(f'\n  --- {llm_id} ({len(grp)} errors) ---')
        # Print up to 3 unique error messages per LLM
        unique_errs = grp['error'].drop_duplicates().head(3).tolist()
        for e in unique_errs:
            print(f'    {e[:300]}')


### 10.1 Inspect pilot outputs

Look at one example output per LLM to verify the JSON shape matches what the prompt asked for (one key per category, with code_ids drawn from the rubric).

In [ ]:
for llm_id in LLMS:
    print(f'\n=== {llm_id} sample output ===')
    sub = pilot_df[(pilot_df['llm_id'] == llm_id) & (pilot_df['parse_ok'])]
    if len(sub) == 0:
        print('  [No successful parses]')
        continue
    row = sub.iloc[0]
    print(f'Question: {row["oe_field"]}')
    print(f'Response: {queue[queue.sona_id == row["sona_id"]]["response"].iloc[0][:200]}...')
    print(f'Coded as:')
    print(json.dumps(row['parsed'], indent=2))

### 10.2 GO gate

Set `PROCEED = True` in the next cell after verifying:
- All three LLMs produced parseable JSON for most responses (parse rate ≥ 90%)
- Output keys match the categories defined in the rubric for each question
- Code values are drawn from the rubric (not hallucinated)
- No glaring errors in the spot-check above

In [ ]:
PROCEED = True  # Set True after reviewing pilot output

if not PROCEED:
    raise RuntimeError(
        'Pilot review gate. Verify the pilot output above looks right. '
        'Then set PROCEED = True and re-run this cell.'
    )

print(f'GO confirmed. Proceeding with full coding pass.')
print(f'Estimated total cost: ${(total_pilot_cost / max(len(pilot_results),1)) * len(queue) * len(LLMS):.2f}')

## 11. Full coding pass

For each (response × LLM) pair: check cache, call LLM if miss, save to cache. Saves partial progress every 25 calls so a crash doesn't lose everything.

In [ ]:
# Master cost tracker (cumulative across all LLMs)
master_cost = 0.0
master_in_tokens = 0
master_out_tokens = 0
n_cache_hits = 0
n_calls_made = 0
n_parse_failures = 0

def code_one(job: Dict[str, Any], llm_id: str) -> Dict[str, Any]:
    """Code one (response × llm). Returns dict to append to results."""
    global master_cost, master_in_tokens, master_out_tokens
    global n_cache_hits, n_calls_made, n_parse_failures

    key = cache_key(llm_id, job['response_hash'], job['oe_field'])
    cached = cache_load(key)

    if cached is not None:
        n_cache_hits += 1
        return {
            'sona_id': job['sona_id'],
            'oe_field': job['oe_field'],
            'condition_label': job['condition_label'],
            'response_hash': job['response_hash'],
            'llm_id': llm_id,
            'rubric_version': RUBRIC_VERSION,
            'rubric_hash': rubric_hash,
            'parsed': cached.get('parsed_json'),
            'parse_ok': cached.get('parsed_json') is not None,
            'in_tok': cached.get('input_tokens', 0),
            'out_tok': cached.get('output_tokens', 0),
            'cost': 0.0,  # already paid
            'cached': True,
        }

    prompt = build_prompt(job['oe_field'], job['response'])
    try:
        r = call_llm(llm_id, prompt)
    except Exception as e:
        return {
            'sona_id': job['sona_id'], 'oe_field': job['oe_field'],
            'condition_label': job['condition_label'], 'response_hash': job['response_hash'],
            'llm_id': llm_id, 'rubric_version': RUBRIC_VERSION, 'rubric_hash': rubric_hash,
            'parsed': None, 'parse_ok': False, 'in_tok': 0, 'out_tok': 0, 'cost': 0.0,
            'cached': False, 'error': str(e),
        }

    n_calls_made += 1
    master_cost += r['cost']
    master_in_tokens += r['input_tokens']
    master_out_tokens += r['output_tokens']

    if r['parsed_json'] is None:
        n_parse_failures += 1

    # Save to cache (even if parse failed — so we know the call happened)
    cache_save(key, {
        'parsed_json': r['parsed_json'],
        'raw_text': r['text'][:5000],  # truncate raw text in cache
        'input_tokens': r['input_tokens'],
        'output_tokens': r['output_tokens'],
        'cached_at': datetime.now().isoformat(timespec='seconds'),
    })

    if master_cost > COST_CEILING_USD:
        raise RuntimeError(f'Cost ceiling ${COST_CEILING_USD} exceeded (${master_cost:.2f}). Aborting.')

    return {
        'sona_id': job['sona_id'], 'oe_field': job['oe_field'],
        'condition_label': job['condition_label'], 'response_hash': job['response_hash'],
        'llm_id': llm_id, 'rubric_version': RUBRIC_VERSION, 'rubric_hash': rubric_hash,
        'parsed': r['parsed_json'], 'parse_ok': r['parsed_json'] is not None,
        'in_tok': r['input_tokens'], 'out_tok': r['output_tokens'], 'cost': r['cost'],
        'cached': False,
    }

print('Coding function ready.')

In [ ]:
# Process LLM by LLM (rather than interleaved) so one LLM's rate-limiting doesn't stall others
all_results = []

# Columns that are safe to serialize for partial-progress checkpoints.
# We drop 'parsed' (the dict of LLM output) because Arrow/parquet cannot handle
# a column that mixes dicts with None values across rows. The full structured
# output is in CACHE_DIR/*.json anyway; the partial save is just for "did we
# get this far" tracking.
PARTIAL_SAVE_COLS = ['sona_id', 'oe_field', 'condition_label', 'response_hash',
                     'llm_id', 'rubric_version', 'rubric_hash', 'parse_ok',
                     'in_tok', 'out_tok', 'cost', 'cached']

for llm_id in LLMS:
    print(f'\n=== Coding pass: {llm_id} ===')
    pre_calls = n_calls_made
    pre_cost = master_cost

    for i, (_, job_row) in enumerate(tqdm(list(queue.iterrows()), desc=llm_id)):
        job = job_row.to_dict()
        result = code_one(job, llm_id)
        all_results.append(result)

        # Periodic partial save (defensive: never let serialization crash the run)
        if (i + 1) % PARTIAL_SAVE_EVERY == 0:
            try:
                partial_df = pd.DataFrame(all_results)
                # Drop the 'parsed' dict column and the 'error' column (if present)
                # before serializing — they're unserializable to parquet but already
                # safe in cache.
                safe_cols = [c for c in PARTIAL_SAVE_COLS if c in partial_df.columns]
                partial_df[safe_cols].to_parquet(
                    CODING_DIR / '_partial_results.parquet', index=False
                )
            except Exception as e:
                # Partial save is best-effort; don't crash the coding pass.
                print(f'  [WARN] Partial save failed at i={i+1}: {e}')

    new_calls = n_calls_made - pre_calls
    new_cost  = master_cost - pre_cost
    print(f'  {llm_id}: {new_calls} new calls (rest cache hits), ${new_cost:.4f} this pass')

print(f'\n=== Coding pass complete ===')
print(f'  Cache hits:      {n_cache_hits:,}')
print(f'  New calls:       {n_calls_made:,}')
print(f'  Parse failures:  {n_parse_failures:,} ({100*n_parse_failures/max(n_calls_made,1):.1f}% of new calls)')
print(f'  Total tokens:    {master_in_tokens:,} in / {master_out_tokens:,} out')
print(f'  Total cost:      ${master_cost:.4f}')


## 12. Expand to long-form binary indicators

The LLM output gives one value per category (a code_id for binary, a list for multilabel). Expand this to one row per (sona_id, oe_field, code_id, llm_id) with a 0/1 value. This is the format Notebook 08 (reliability) and Notebook 09 (integration) consume.

In [ ]:
def expand_codes(parsed: Optional[Dict[str, Any]], question_id: str) -> List[Dict[str, Any]]:
    """Expand a category-keyed JSON output to per-theme binary indicators."""
    if parsed is None:
        return []
    qdata = rubric['questions'][question_id]
    expanded = []

    for t in qdata['themes']:
        cat_id = t['category']
        code_id = t['code_id']
        cat_info = qdata['categories'].get(cat_id, {})
        coding_type = cat_info.get('coding_type', 'multilabel')

        cat_value = parsed.get(cat_id)

        if coding_type == 'binary':
            value = 1 if cat_value == code_id else 0
        else:  # multilabel
            if isinstance(cat_value, list):
                value = 1 if code_id in cat_value else 0
            else:
                value = 0

        expanded.append({
            'category': cat_id,
            'code_id': code_id,
            'coding_type': coding_type,
            'value': value,
        })
    return expanded

# Build long-form output
long_rows = []
for r in all_results:
    if not r['parse_ok']:
        continue
    expanded = expand_codes(r['parsed'], r['oe_field'])
    for e in expanded:
        long_rows.append({
            'sona_id': r['sona_id'],
            'condition_label': r['condition_label'],
            'oe_field': r['oe_field'],
            'response_hash': r['response_hash'],
            'llm_id': r['llm_id'],
            'rubric_version': r['rubric_version'],
            'category': e['category'],
            'code_id': e['code_id'],
            'coding_type': e['coding_type'],
            'value': e['value'],
            'confidence': (r['parsed'] or {}).get('_confidence'),
        })

long_df = pd.DataFrame(long_rows)
print(f'Long-form output: {len(long_df):,} rows '
      f'({long_df["sona_id"].nunique()} participants × '
      f'{long_df["oe_field"].nunique()} questions × '
      f'{long_df["code_id"].nunique()} codes × '
      f'{long_df["llm_id"].nunique()} LLMs)')

## 13. Per-LLM raw outputs (for transparency / debugging)

Save each LLM's raw category-level outputs separately. Useful for inspecting disagreements later.

In [ ]:
for llm_id in LLMS:
    sub = [r for r in all_results if r['llm_id'] == llm_id and r['parse_ok']]
    rows = []
    for r in sub:
        row = {
            'sona_id': r['sona_id'],
            'oe_field': r['oe_field'],
            'condition_label': r['condition_label'],
            'response_hash': r['response_hash'],
        }
        # Flatten category outputs
        for cat_id, cat_val in (r['parsed'] or {}).items():
            row[f'cat_{cat_id}'] = json.dumps(cat_val) if cat_val is not None else None
        rows.append(row)

    if rows:
        df = pd.DataFrame(rows)
        out = PER_LLM_DIR / f'{llm_id}.parquet'
        df.to_parquet(out, index=False)
        print(f'Wrote {out.name}: {len(df):,} rows')

## 14. Sanity statistics

Per-code prevalence by LLM, plus a quick LLM-by-LLM agreement spot-check on a few high-prevalence codes.

In [ ]:
# Prevalence by code × LLM
prevalence = (
    long_df
    .groupby(['oe_field', 'category', 'code_id', 'llm_id'])['value']
    .agg(['mean', 'sum', 'count'])
    .reset_index()
    .rename(columns={'mean': 'prevalence', 'sum': 'n_endorsed', 'count': 'n_total'})
)
prevalence['prevalence_pct'] = (prevalence['prevalence'] * 100).round(1)

# Pivot for readability: one row per (oe_field, category, code), columns = LLMs
prevalence_wide = (
    prevalence
    .pivot_table(index=['oe_field', 'category', 'code_id'],
                 columns='llm_id', values='prevalence_pct')
    .round(1)
)
print('Per-code prevalence (% of responses where the LLM coded the theme as 1):\n')
for q in TARGET_QUESTIONS:
    print(f'\n--- {q} ---')
    sub = prevalence_wide.loc[q] if q in prevalence_wide.index.get_level_values(0) else None
    if sub is None or sub.empty:
        print('  (no data)')
        continue
    print(sub.to_string())

In [ ]:
# Quick agreement check: for each code, fraction of responses where all 3 LLMs agree
agreement = (
    long_df
    .pivot_table(index=['sona_id', 'oe_field', 'code_id'], columns='llm_id', values='value')
    .reset_index()
)
llm_cols = list(LLMS.keys())
llm_cols_present = [c for c in llm_cols if c in agreement.columns]
if len(llm_cols_present) == 3:
    all_three = agreement[llm_cols_present].dropna()
    all_agree = (all_three.nunique(axis=1) == 1).mean()
    print(f'\nAcross all (response × code) cells with all 3 LLMs coding successfully:')
    print(f'  N cells: {len(all_three):,}')
    print(f'  Fraction where all 3 LLMs agree (0=0=0 or 1=1=1): {all_agree:.3f}')
    print(f'  (Note: this includes the unanimous-zero majority. Cohen\'s κ in Notebook 08 will be more informative.)')

## 15. Save outputs

In [ ]:
# Long-form coded data
long_path = CODING_DIR / 'oe_coded_long.parquet'
long_df.to_parquet(long_path, index=False)
print(f'Wrote {long_path.name}: {len(long_df):,} rows, {long_path.stat().st_size/1024:.1f} KB')

# Manifest
manifest = {
    'generated_at': datetime.now().isoformat(timespec='seconds'),
    'rubric_version': RUBRIC_VERSION,
    'rubric_hash': rubric_hash,
    'rubric_path': str(rubric_path.relative_to(PROJECT_ROOT)),
    'llms': {llm_id: {'model': cfg['model'], 'family': cfg['family']}
             for llm_id, cfg in LLMS.items()},
    'queue_size': len(queue),
    'cache_hits': int(n_cache_hits),
    'new_calls': int(n_calls_made),
    'parse_failures': int(n_parse_failures),
    'total_tokens_in': int(master_in_tokens),
    'total_tokens_out': int(master_out_tokens),
    'total_cost_usd': round(master_cost, 4),
    'long_form_rows': int(len(long_df)),
    'unique_responses_coded': int(long_df['sona_id'].nunique()),
    'codes_per_question': {
        q: int(long_df[long_df['oe_field']==q]['code_id'].nunique())
        for q in TARGET_QUESTIONS
    },
}
manifest_path = CODING_DIR / 'coding_manifest.json'
with open(manifest_path, 'w') as f:
    json.dump(manifest, f, indent=2)
print(f'Wrote {manifest_path.name}')

# Clean up partial-progress file
partial = CODING_DIR / '_partial_results.parquet'
if partial.exists():
    partial.unlink()

## 16. Wrap-up

After this completes, the OE responses are fully coded. Notebook 08 (reliability) will compute κ between every LLM pair on every code; Notebook 09 (integration) will produce participant-level NLP features.

If parse failure rate is >5%, inspect `_cache/` for the failed responses — usually the issue is a single-line response that the LLM tried to "explain" rather than code, or markdown fences the parser missed. Fix the prompt and re-run; cached successful calls won't be re-charged.

In [ ]:
print('=' * 70)
print('NOTEBOOK 04 COMPLETE')
print('=' * 70)
print(f'Rubric version:  {RUBRIC_VERSION} (hash {rubric_hash})')
print(f'Responses coded: {long_df["sona_id"].nunique()} participants × '
      f'{long_df["oe_field"].nunique()} questions = '
      f'{long_df.groupby(["sona_id","oe_field"]).ngroups} response-units')
print(f'LLMs:            {", ".join(LLMS.keys())}')
print(f'Cost this run:   ${master_cost:.4f} ({n_calls_made:,} new calls, {n_cache_hits:,} cache hits)')
print(f'Parse failures:  {n_parse_failures:,}')
print()
print(f'Outputs:')
print(f'  {long_path}')
print(f'  {manifest_path}')
print(f'  {PER_LLM_DIR}/ (per-LLM raw category outputs)')
print()
print('Next: Notebook 05 (cross-family kappa + consensus aggregation).')
print('      Then Notebooks 06 (topic modeling), 07 (sentiment), and 08 (turn-level chat coding).')
